In [0]:
from scripts.utils import silver_utils
print(dir(silver_utils))

In [0]:
from pyspark.sql.functions import (
    col, when, sum, lit, trim, lower, upper,
    to_date, current_timestamp,round
)
from datetime import datetime
import uuid,logging

In [0]:
df=spark.read.table("workspace.ecommerce_bronze.df_customers")
logging.info('table read successfully')

In [0]:
silver_utils.check_schema(df)

In [0]:
df =silver_utils.cast_types(df,{
    "customer_zip_code_prefix": "integer",
    "customer_city": "string",
    "customer_state": "string",
    "customer_id": "string"
})

In [0]:
silver_utils.null_profiling(df,"df_customers")
df= silver_utils.handle_nulls_drop(df,drop_cols=['customer_id',"customer_city"])


In [0]:
df = silver_utils.handle_duplicates(df,['customer_id'])

In [0]:
df = silver_utils.standardize_strings(df,{
    "customer_city": lambda c : lower(trim(c)),
    "customer_state": lambda c : upper(trim(c)),
    "customer_id": lambda c : trim(c)
})

In [0]:

checks = [
    # nulls
    (
        "customer_id",
        "customer_id not null",
        col("customer_id").isNotNull()
    ),
    (
        "customer_zip_code_prefix",
        "zip not null",
        col("customer_zip_code_prefix").isNotNull()
    ),
    (
        "customer_city",
        "city not null",
        col("customer_city").isNotNull()
    ),
    (
        "customer_state",
        "state not null",
        col("customer_state").isNotNull()
    ),

    #duplicates
    (
        "customer_id",
        "customer_id is unique",
        col("customer_id").isNotNull()  # see note below
    ),

    # standardization
    (
        "customer_state",
        "state is 2-letter uppercase",
        col("customer_state").rlike(r"^[A-Z]{2}$")
    ),
    (
        "customer_city",
        "city not blank or whitespace",
        trim(col("customer_city")) != ""
    ),
    (
        "customer_zip_code_prefix",
        "zip is 5 digits",
        (col("customer_zip_code_prefix") >= 10000) &
        (col("customer_zip_code_prefix") <= 99999)
    ),
]

dq_table = silver_utils.build_dq_table(
    spark=spark,
    df=df,
    checks=checks,
    table_name="df_customers",
    warn_threshold=5.0
)

dq_table.write.mode("overwrite").saveAsTable("workspace.ecommerce_silver.df_customers_dq")
dq_saved = spark.read.table("workspace.ecommerce_silver.df_customers_dq")
display(dq_saved)

In [0]:
df =silver_utils.add_silver_metadata(df)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.ecommerce_silver.df_customers")
print("saved successfully !")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.ecommerce_silver.df_customers LIMIT 10

In [0]:
dbutils.library.restartPython()
